In [1]:
import duckdb
import numpy as np
import pandas as pd

pdir = "../../data/parquet"

# one row per cell line, one column per gene
expr = duckdb.sql(f"""
    SELECT ACH_ID, ensembl_id, median(log2_tpm_plus1) AS v
    FROM read_parquet('{pdir}/fact_expression_depmap.parquet')
    GROUP BY ACH_ID, ensembl_id
""").df()

print("long form:", expr.shape)

wide = expr.pivot(index="ACH_ID", columns="ensembl_id", values="v")
print("wide:", wide.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

long form: (79182702, 3)
wide: (1479, 53538)


In [2]:
# drop genes with no variation, then keep the most variable
wide = wide.fillna(0)
variances = wide.var()
top_genes = variances.nlargest(2000).index
expr_matrix = wide[top_genes]

print("reduced:", expr_matrix.shape)
print("variance range kept:", variances[top_genes].min().round(3), "to", variances[top_genes].max().round(3))

reduced: (1479, 2000)
variance range kept: 2.676 to 19.727


In [3]:
dim = pd.read_parquet(f"{pdir}/dim_cell_lines.parquet")
lineage = dim.set_index("ach_id")["lineage"]

expr_matrix = expr_matrix.copy()
expr_matrix["lineage"] = expr_matrix.index.map(lineage)
print("with lineage:", expr_matrix["lineage"].notna().sum(), "of", len(expr_matrix))

with lineage: 1412 of 1479


In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

# same test used on the contextual features, so the two are directly comparable
def lineage_test(matrix, feat_cols, label, n_pairs=2000, seed=0):
    rng = np.random.default_rng(seed)
    m = matrix.dropna(subset=["lineage"]).reset_index(drop=True)
    X = m[feat_cols].fillna(0).values
    sims = cosine_similarity(X)

    # sample random pairs and split by whether they share a lineage
    same, diff = [], []
    for _ in range(n_pairs):
        i, j = rng.integers(0, len(m), 2)
        if i == j:
            continue
        (same if m.loc[i, "lineage"] == m.loc[j, "lineage"] else diff).append(sims[i, j])

    print(f"{label}")
    print(f"  same lineage: mean {np.mean(same):.4f}  (n={len(same)})")
    print(f"  diff lineage: mean {np.mean(diff):.4f}  (n={len(diff)})")
    print(f"  gap: {np.mean(same) - np.mean(diff):+.4f}")

gene_cols = [c for c in expr_matrix.columns if c != "lineage"]

lineage_test(expr_matrix, gene_cols, "expression, raw")

# z-scored so no single high-expression gene dominates the distance
scaled = expr_matrix.copy()
scaled[gene_cols] = StandardScaler().fit_transform(scaled[gene_cols])
lineage_test(scaled, gene_cols, "expression, z-scored")

expression, raw
  same lineage: mean 0.7845  (n=114)
  diff lineage: mean 0.6796  (n=1886)
  gap: +0.1049
expression, z-scored
  same lineage: mean 0.2872  (n=114)
  diff lineage: mean -0.0195  (n=1886)
  gap: +0.3067


In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

pdir = "../../data/parquet"

def scale(wide):
    wide = wide.loc[:, wide.columns.notna()]
    wide.columns = wide.columns.astype(str)
    wide = wide.fillna(wide.median())
    return pd.DataFrame(StandardScaler().fit_transform(wide),
                        index=wide.index, columns=wide.columns)

# protein matrix
prot_long = duckdb.sql(f"""
    SELECT CAST(ach_id AS VARCHAR) AS ach_id,
           CAST(ensembl_id AS VARCHAR) AS ensembl_id,
           median(protein_intensity) AS v
    FROM read_parquet('{pdir}/fact_proteomics.parquet')
    WHERE protein_intensity IS NOT NULL
    GROUP BY ach_id, ensembl_id
""").df()
prot = scale(prot_long.pivot(index="ach_id", columns="ensembl_id", values="v"))
print("protein matrix:", prot.shape)

# expression matrix restricted to the same cell lines, so the comparison is like for like
shared = sorted(set(prot.index) & set(expr_matrix.index))
print("cell lines with both:", len(shared))

gene_cols = [c for c in expr_matrix.columns if c != "lineage"]
e = StandardScaler().fit_transform(expr_matrix.loc[shared, gene_cols])
p = prot.loc[shared]

e_sims = cosine_similarity(e)
p_sims = cosine_similarity(p)

# compare the two on the same pairs
iu = np.triu_indices(len(shared), k=1)
ev, pv = e_sims[iu], p_sims[iu]

print(f"\nexpression: mean {ev.mean():+.4f}  sd {ev.std():.4f}")
print(f"protein:    mean {pv.mean():+.4f}  sd {pv.std():.4f}")
print(f"correlation between the two: {np.corrcoef(ev, pv)[0,1]:.4f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

protein matrix: (375, 11996)
cell lines with both: 369

expression: mean -0.0016  sd 0.1813
protein:    mean -0.0023  sd 0.1181
correlation between the two: 0.4830
